# Random Guide-Selection Strategies

Compares different random strategies for selecting guide sets.

`driver.py` now writes **one `results.parquet` per run**, where a run is a single
strategy/config (see `config.json`). So each entry in `RUNS` below is one run
directory, and the strategies are overlaid across runs (what used to be
"sampling modes" within a single run).

In [ ]:
import json
import math

import numpy as np
import polars as pl
import matplotlib.pyplot as plt

from helpers import (
    find_latest_run,
    load_driver_run,
    load_baseline,
    stop_reason_category,
    compute_goal_reach,
    compute_goal_reach_matrix,
    compute_war_agg,
)
from plots import (
    configure_k_axis,
    finish_fig,
    plot_metric_boxplots,
    annotate_k_points,
    plot_mean_with_best_dotted,
    split_modes,
)

plt.rcParams.update({"figure.dpi": 240, "figure.figsize": (12, 8)})


# plt.xkcd()

In [ ]:
# Configuration

# Each run directory holds a single strategy/config (driver.py writes one
# results.parquet per run). List the runs you want to overlay here as
# (display_name, run-directory pattern); the strategy/k come from the data.
RUNS = [
    ("no-replacement-count", "run.5"),
    # ("with-replacement-count", ""),
    # ("no-replacement-naive", ""),
    # ("with-replacement-naive", ""),
    # ("smallest-overall", ""),
    # ("smallest-novel", ""),
]

# smallest_* strategies emit a single k=1 point per goal (no k sweep); everything
# else sweeps k. Used to decide marker style and skip per-k best/std overlays.
SINGLE_POINT_STRATEGIES = {"smallest_overall", "smallest_novel"}

PALETTE = plt.cm.tab10.colors  # type: ignore[attr-defined]
MARKER_CYCLE = ["o", "s", "D", "^", "v", "P", "X", "*"]

METRIC_BEST_AGG = {
    "iters": "min",
    "nodes": "min",
    "classes": "min",
    "nodes_per_class": "max",
    "total_time": "min",
}


def mode_style(i, is_single_point=False):
    return {
        "color": PALETTE[i % len(PALETTE)],
        "marker": "x" if is_single_point else MARKER_CYCLE[i % len(MARKER_CYCLE)],
        "linestyle": "none" if is_single_point else "-",
        "markersize": 10 if is_single_point else 6,
        "linewidth": 0 if is_single_point else 1.5,
        "zorder": 5 if is_single_point else 2,
    }

In [ ]:
# ── Load & parse ──────────────────────────────────────────────────────
# One run == one strategy. We overlay the runs as "modes", keeping the old
# dataset_frames[ds_name][mode_name] -> DataFrame shape (single implicit
# dataset) so the downstream plotting cells stay unchanged.
DS_NAME = "random-strategies"
ds_names = [DS_NAME]

# Built from RUNS: (mode_name, run_dir, is_single_point). Replaces the old
# file-prefix SAMPLING_MODES; the middle element is now the run directory.
SAMPLING_MODES = []
dataset_frames = {DS_NAME: {}}
dataset_n = {}  # DS_NAME -> (n_goals, n_trials) for annotating the multi-modes

mode_dfs = dataset_frames[DS_NAME]
ref = None
for mode_name, pattern in RUNS:
    run_dir = find_latest_run(pattern)
    config = json.loads((run_dir / "config.json").read_text())
    strategy = config["strategy"]
    is_single = strategy in SINGLE_POINT_STRATEGIES

    df = load_driver_run(run_dir)
    # Drop empty-pool legs (k=0, nothing to sample): not a real guide-set size.
    df = df.filter(pl.col("k") > 0)
    df = df.with_columns((pl.col("nodes") / pl.col("classes")).alias("nodes_per_class"))
    mode_dfs[mode_name] = df
    SAMPLING_MODES.append((mode_name, run_dir, is_single))

    n_reached = df["reached"].sum()
    print(
        f"{mode_name}: {run_dir.name} strategy={strategy} — {len(df)} rows, "
        f"k={sorted(df['k'].unique().to_list())}, {n_reached} reached"
    )

    # driver.py early-stops a pair on first reach, so the row count per (goal, k)
    # varies. Expose (n_goals, configured attempts) for the plot annotations
    # instead of asserting a constant trial count.
    if not is_single:
        n_goals = df["goal"].n_unique()
        n_trials = config["attempts"]
        if ref is None:
            ref = (n_goals, n_trials)

dataset_n[DS_NAME] = ref

# Union of k values across all multi-point runs (single-point runs only have k=1).
k_values = sorted(
    {
        k
        for mode_name, _run_dir, is_single in SAMPLING_MODES
        if not is_single
        for k in mode_dfs[mode_name]["k"].unique().to_list()
    }
)

print(f"\nk values: {k_values}")
print(f"(n_goals, n_trials) [attempts] for multi-point modes: {dataset_n}")

In [ ]:
# ── Baselines (full eqsat, no guides) ────────────────────────────────────
# Each run's config.json points at its seed-terms folder, whose terms.json
# carries a per-seed `goal_egraph` block (the full eqsat on the seed with no
# guides). `load_baseline` averages those over the Ok seeds. The overlaid runs
# share a seed set, so we average their per-run baselines into one dataset
# baseline (mode-independent unguided cost).
baselines = {}
for ds_name in ds_names:
    per_mode = [load_baseline(run_dir) for _mode_name, run_dir, _is_single in SAMPLING_MODES]
    metrics = per_mode[0].keys()
    baselines[ds_name] = {m: sum(b[m] for b in per_mode) / len(per_mode) for m in metrics}

print("Baselines (full eqsat, no guides):")
for ds_name, vals in baselines.items():
    print(f"  {ds_name}: nodes={vals['nodes']:.1f}, time={vals['total_time']:.6f}s")

## Metrics vs k

In [ ]:
BASELINE_METRICS = {"iters", "nodes", "classes", "nodes_per_class", "total_time"}

METRICS = ["iters", "nodes", "classes", "nodes_per_class", "total_time"]
n_metrics = len(METRICS)
n_cols = 3
n_rows = math.ceil(n_metrics / n_cols)

for ds_name in ds_names:
    mode_dfs = dataset_frames[ds_name]
    mode_dfs = {k: v.filter(pl.col("reached")) for k, v in mode_dfs.items()}
    n_goals, n_trials = dataset_n[ds_name]
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 5 * n_rows))
    axes_flat = axes.flat

    for metric, ax in zip(METRICS, axes_flat):
        best_agg = METRIC_BEST_AGG[metric]

        for i, (mode_name, _run_dir, is_single) in enumerate(SAMPLING_MODES):
            df = mode_dfs[mode_name]
            style = mode_style(i, is_single)

            if is_single:
                agg = (
                    df.group_by("k")
                    .agg(pl.col(metric).mean().alias("mean"))
                    .sort("k")
                    .drop_nulls("mean")
                )
                for row in agg.iter_rows(named=True):
                    ax.plot(row["k"], row["mean"], label=mode_name, **style)
            else:
                agg = (
                    df.group_by("k")
                    .agg(
                        pl.col(metric).mean().alias("mean"),
                        pl.col(metric).std().alias("std"),
                    )
                    .sort("k")
                )
                ks = agg["k"]
                means = agg["mean"]
                stds = agg["std"].fill_null(0)
                ax.fill_between(ks, means - stds, means + stds, alpha=0.15, color=style["color"])

                per_seed_best = (
                    df.group_by("k", "seed")
                    .agg(getattr(pl.col(metric), best_agg)().alias("seed_best"))
                    .group_by("k")
                    .agg(pl.col("seed_best").mean().alias("best"))
                    .sort("k")
                )
                plot_mean_with_best_dotted(
                    ax,
                    ks,
                    means,
                    per_seed_best["best"],
                    style,
                    label=mode_name,
                    best_label=f"{mode_name} best ({best_agg})",
                )

        if baselines.get(ds_name) and metric in BASELINE_METRICS:
            bval = baselines[ds_name][metric]
            ax.axhline(
                bval, color="black", linestyle="--", linewidth=1.2, alpha=0.7, label="baseline"
            )

        ax.set_xlabel("k (number of guides)")
        metric_label = {
            "nodes": "nodes (incl. guide egraph)",
            "total_time": "total_time (incl. guide eqsat)",
        }.get(metric, metric)
        ax.set_ylabel(metric_label)
        ax.set_title(f"{metric_label} vs k (mean ± std)")
        ax.legend(fontsize=7)
        configure_k_axis(ax, k_values)

    for ax in list(axes_flat)[n_metrics:]:
        ax.set_visible(False)

    finish_fig(fig, "Metrics vs k", ds_name, n_goals=n_goals, n_trials=n_trials)

## Reachability rate vs k

In [ ]:
# Distinct linestyles for stop-reason categories (right plot, multi-modes only)
STOP_REASON_LINESTYLES = {
    "Saturated": "solid",
    "IterationLimit": (0, (4, 2)),
    "NodeLimit": (0, (1, 1)),
    "TimeLimit": (0, (3, 1, 1, 1)),
    "Other": (0, (5, 1, 1, 1, 1, 1)),
}

for ds_name in ds_names:
    mode_dfs = dataset_frames[ds_name]
    n_goals, n_trials = dataset_n[ds_name]
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for i, (mode_name, _run_dir, is_single) in enumerate(SAMPLING_MODES):
        df = mode_dfs[mode_name]
        style = mode_style(i, is_single)
        label_suffix = f" ({n_goals}×{n_trials})" if not is_single else ""

        # Per-(k, stop_reason) breakdown of unreached legs. `stop_reason` is null
        # on reached legs, so categorize only the failures.
        unreached_breakdown = (
            df.filter(~pl.col("reached"))
            .with_columns(
                pl.col("stop_reason")
                .map_elements(stop_reason_category, return_dtype=pl.String)
                .alias("stop_reason_cat")
            )
            .group_by("k", "stop_reason_cat")
            .agg(pl.len().alias("count"))
            .join(df.group_by("k").agg(pl.len().alias("total")), on="k")
            .with_columns((pl.col("count") / pl.col("total")).alias("rate"))
            .sort("k")
        )

        rates = (
            df.group_by("k")
            .agg(
                pl.col("reached").mean().alias("reached_rate"),
                (~pl.col("reached")).mean().alias("unreached_rate"),
            )
            .sort("k")
        )
        ks = rates["k"]

        ax = axes[0]
        ax.plot(
            ks, rates["reached_rate"] * 100, label=f"{mode_name} reached{label_suffix}", **style
        )

        ax = axes[1]
        ax.plot(
            ks,
            rates["unreached_rate"] * 100,
            label=f"{mode_name} unreached{label_suffix}",
            **style,
        )
        if not is_single:
            for cat, ls in STOP_REASON_LINESTYLES.items():
                sub = unreached_breakdown.filter(pl.col("stop_reason_cat") == cat).sort("k")
                if sub.is_empty():
                    continue
                ax.plot(
                    sub["k"],
                    sub["rate"] * 100,
                    color=style["color"],
                    linestyle=ls,
                    linewidth=1.2,
                    alpha=0.7,
                    label=f"{mode_name} unreached/{cat}{label_suffix}",
                )

    axes[0].set_xlabel("k (number of guides)")
    axes[0].set_ylabel("Rate (%)")
    axes[0].set_title("Reachability rate vs k")
    axes[0].legend(fontsize=8)
    configure_k_axis(axes[0], k_values)
    axes[0].set_ylim(-5, 105)

    axes[1].set_xlabel("k (number of guides)")
    axes[1].set_ylabel("Rate (%)")
    axes[1].set_title(
        "Failure breakdown vs k\n(solid = total unreached; per-stop-reason lines split it)"
    )
    axes[1].legend(fontsize=7)
    configure_k_axis(axes[1], k_values)
    axes[1].set_ylim(-5, 105)

    finish_fig(fig, "Reachability vs k", ds_name, n_goals=n_goals, n_trials=n_trials)

## Box plots: iterations by strategy at each k

In [ ]:
for ds_name in ds_names:
    mode_dfs = dataset_frames[ds_name]
    n_goals, n_trials = dataset_n[ds_name]
    # Tag each run with its display mode name so we group by mode (one strategy
    # per run means the raw `strategy` column can't distinguish overlaid runs).
    combined = pl.concat(
        [
            v.filter(pl.col("reached")).with_columns(pl.lit(name).alias("mode"))
            for name, v in mode_dfs.items()
        ]
    )
    mode_names = [m for m, _p, _s in SAMPLING_MODES]
    plot_metric_boxplots(
        combined,
        ds_name,
        "iters",
        "iters",
        k_values,
        mode_names,
        PALETTE,
        n_goals=n_goals,
        n_trials=n_trials,
        group_col="mode",
    )

## Box plots: nodes by strategy at each k

Node count includes the guide egraph (i.e. `max(trial_nodes, guide_egraph_nodes)`).

In [ ]:
for ds_name in ds_names:
    mode_dfs = dataset_frames[ds_name]
    n_goals, n_trials = dataset_n[ds_name]
    combined = pl.concat(
        [
            v.filter(pl.col("reached")).with_columns(pl.lit(name).alias("mode"))
            for name, v in mode_dfs.items()
        ]
    )
    mode_names = [m for m, _p, _s in SAMPLING_MODES]
    plot_metric_boxplots(
        combined,
        ds_name,
        "nodes",
        "nodes (incl. guide egraph)",
        k_values,
        mode_names,
        PALETTE,
        n_goals=n_goals,
        n_trials=n_trials,
        group_col="mode",
    )

## Summary table

In [ ]:
from IPython.display import display

for ds_name in ds_names:
    print(f"\n=== {ds_name} ===")
    combined = pl.concat(
        [v.with_columns(pl.lit(name).alias("mode")) for name, v in dataset_frames[ds_name].items()]
    )
    summary = (
        combined.group_by("mode", "k")
        .agg(
            pl.col("iters").mean().round(2).alias("avg_iters"),
            pl.col("nodes").mean().round(0).alias("avg_nodes"),
            pl.col("classes").mean().round(0).alias("avg_classes"),
            pl.col("nodes_per_class").mean().round(2).alias("avg_nodes_per_class"),
            pl.col("total_time").mean().round(4).alias("avg_time_s"),
            pl.col("reached").mean().round(3).alias("reached_rate"),
            (~pl.col("reached")).mean().round(3).alias("unreached_rate"),
            pl.col("reached").sum().alias("n_reached"),
            pl.len().alias("n_legs"),
        )
        .sort("k", "mode")
    )
    display(summary)

## Per-goal analysis

In [ ]:
for ds_name in ds_names:
    mode_dfs = dataset_frames[ds_name]
    for mode_name, _run_dir, _is_single in SAMPLING_MODES:
        df = mode_dfs[mode_name]
        goal_reach = compute_goal_reach(df)
        print(
            f"{ds_name} / {mode_name} — {goal_reach.shape[0]} goal×k combinations, {goal_reach['goal'].n_unique()} unique goals"
        )

## Goal difficulty distribution

In [ ]:
for ds_name in ds_names:
    mode_dfs = dataset_frames[ds_name]
    n_goals, n_trials = dataset_n[ds_name]

    multi_modes, single_modes = split_modes(SAMPLING_MODES)

    n_multi = len(multi_modes)
    hist_rows = math.ceil(n_multi / 2)
    total_rows = hist_rows + (1 if single_modes else 0)

    height_ratios = [3] * hist_rows + ([1] if single_modes else [])
    fig = plt.figure(figsize=(16, 3 * hist_rows + (1.2 if single_modes else 0)))
    gs = fig.add_gridspec(
        total_rows,
        3,
        height_ratios=height_ratios,
        width_ratios=[1, 1, 2],
        wspace=0.35,
        hspace=0.4,
    )

    # Multi-strategy histograms
    for slot, (i, mode_name, _fp, is_single) in enumerate(multi_modes):
        style = mode_style(i, is_single)
        ax = fig.add_subplot(gs[slot // 2, slot % 2])
        df = mode_dfs[mode_name]
        goal_reach = compute_goal_reach(df)
        avg_reach = (
            goal_reach.group_by("goal")
            .agg(pl.col("cond_reach_rate").mean().alias("avg_reach"))["avg_reach"]
            .to_numpy()
            * 100
        )
        ax.hist(avg_reach, bins=20, range=(0, 100), color=style["color"], alpha=0.85)
        ax.set_xlim(0, 100)
        ax.set_title(mode_name, fontsize=9)
        ax.set_ylabel("# goals", fontsize=8)
        if slot >= n_multi - 2:
            ax.set_xlabel("Avg reachability (%)", fontsize=8)

    # Single-strategy bars combined in one wide panel spanning cols 0-1
    if single_modes:
        ax_single = fig.add_subplot(gs[hist_rows, 0:2])
        for j, (i, mode_name, _fp, is_single) in enumerate(single_modes):
            style = mode_style(i, is_single)
            df = mode_dfs[mode_name]
            goal_reach = compute_goal_reach(df)
            n_single = len(df)
            frac = (
                goal_reach.group_by("goal")
                .agg(pl.col("cond_reach_rate").mean().alias("avg_reach"))["avg_reach"]
                .mean()
                * 100
            )  # pyright: ignore[reportOperatorIssue]
            ax_single.barh([j], [frac], color=style["color"], alpha=0.85, height=0.5)  # pyright: ignore[reportArgumentType]
            ax_single.barh([j], [100 - frac], left=[frac], color="lightgray", alpha=0.6, height=0.5)  # pyright: ignore[reportOperatorIssue, reportArgumentType]
            ax_single.text(
                frac / 2,  # pyright: ignore[reportOperatorIssue,reportArgumentType]
                j,
                f"{frac:.0f}%  (n={n_single})",
                ha="center",
                va="center",
                fontsize=9,
                color="white" if frac > 15 else "black",  # pyright: ignore[reportOperatorIssue]
            )
        ax_single.set_xlim(0, 100)
        ax_single.set_yticks(range(len(single_modes)))
        ax_single.set_yticklabels([name for _, name, _, _ in single_modes], fontsize=8)
        ax_single.set_xlabel("Goals reached (%)", fontsize=8)

    # CDF — only multi strategies, spanning all rows on the right
    ax_cdf = fig.add_subplot(gs[:, 2])
    for i, mode_name, _fp, is_single in multi_modes:
        style = mode_style(i, is_single)
        df = mode_dfs[mode_name]
        goal_reach = compute_goal_reach(df)
        mode_k_values = sorted(df["k"].unique().to_list())
        for j, k in enumerate(mode_k_values):
            reach_at_k = goal_reach.filter(pl.col("k") == k).sort("reach_rate")["reach_rate"]
            ax_cdf.plot(
                reach_at_k,
                np.linspace(0, 1, len(reach_at_k)),
                label=f"{mode_name} k={k}",
                color=style["color"],
                linestyle=["-", "--", ":", "-."][j % 4],
            )
    ax_cdf.set_xlabel("Per-goal reachability rate")
    ax_cdf.set_ylabel("Fraction of goals (CDF)")
    ax_cdf.set_title("CDF of per-goal reachability by k")
    ax_cdf.legend(fontsize=7)

    finish_fig(fig, "Goal difficulty distribution", ds_name, n_goals=n_goals, n_trials=n_trials)

## Goal-level reachability heatmap

In [ ]:
for ds_name in ds_names:
    mode_dfs = dataset_frames[ds_name]
    n_goals, n_trials = dataset_n[ds_name]
    multi_modes, _ = split_modes(SAMPLING_MODES)
    n_modes = len(multi_modes)
    n_cols_hm = min(4, n_modes)
    n_rows_hm = math.ceil(n_modes / n_cols_hm)

    fig, axes = plt.subplots(
        n_rows_hm,
        n_cols_hm,
        figsize=(3 * n_cols_hm + 1, 5 * n_rows_hm),
        squeeze=False,
    )

    im = None
    rightmost_ax = None
    for si, (_i, mode_name, _file_prefix, _is_single) in enumerate(multi_modes):
        ax = axes[si // n_cols_hm][si % n_cols_hm]
        df = mode_dfs[mode_name]
        mode_k_values = sorted(df["k"].unique().to_list())

        _goal_order, matrix = compute_goal_reach_matrix(df, mode_k_values)

        im = ax.imshow(matrix, aspect="auto", cmap="RdYlGn", vmin=0, vmax=1)
        ax.set_xticks(range(len(mode_k_values)))
        ax.set_xticklabels([str(k) for k in mode_k_values])
        ax.set_xlabel("k (number of guides)")
        ax.set_ylabel("Goal (sorted by avg reachability, hardest at top)")
        ax.set_yticks([])
        ax.set_title(mode_name)
        rightmost_ax = ax

    for si in range(n_modes, n_rows_hm * n_cols_hm):
        axes[si // n_cols_hm][si % n_cols_hm].set_visible(False)

    if im is not None and rightmost_ax is not None:
        cbar = fig.colorbar(
            im, ax=rightmost_ax, orientation="vertical", label="Reachability rate (%)"
        )
        cbar.set_ticks([0, 0.25, 0.5, 0.75, 1.0])
        cbar.set_ticklabels(["0%", "25%", "50%", "75%", "100%"])

    fig.subplots_adjust(left=0.05, top=0.88, bottom=0.1, wspace=0.3)

    finish_fig(fig, "Goal-level reachability heatmap", ds_name, n_goals=n_goals, n_trials=n_trials)

## Goal reachability: coverage and diminishing returns

In [ ]:
for ds_name in ds_names:
    mode_dfs = dataset_frames[ds_name]
    n_goals, n_trials = dataset_n[ds_name]
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # ── Left: Fraction of goals with ≥1 successful trial vs k ─────────
    ax = axes[0]
    for i, (mode_name, _run_dir, is_single) in enumerate(SAMPLING_MODES):
        style = mode_style(i, is_single)
        df = mode_dfs[mode_name]
        label = mode_name if is_single else f"{mode_name} ({n_goals}×{n_trials})"

        any_success_per_goal_k = df.group_by("goal", "k").agg(
            pl.col("reached").any().alias("any_success")
        )
        frac = (
            any_success_per_goal_k.group_by("k")
            .agg(pl.col("any_success").mean().alias("frac_goals"))
            .sort("k")
        )
        ax.plot(frac["k"], frac["frac_goals"] * 100, label=label, **style)

    ax.set_xlabel("k (number of guides)")
    ax.set_ylabel("Goals with ≥ 1 successful trial (%)")
    ax.set_title(f"Fraction of goals with at least one successful trial vs k [{ds_name}]")
    ax.legend()
    configure_k_axis(ax, k_values)
    ax.set_ylim(-5, 105)

    # ── Right: Diminishing returns — per-goal reachability vs k ───────
    ax = axes[1]
    for i, (mode_name, _run_dir, is_single) in enumerate(SAMPLING_MODES):
        style = mode_style(i, is_single)
        df = mode_dfs[mode_name]
        goal_reach = compute_goal_reach(df)
        mode_k_values = sorted(df["k"].unique().to_list())

        reach_at_k1 = (
            goal_reach.filter(pl.col("k") == mode_k_values[0])
            .select("goal", "reach_rate")
            .rename({"reach_rate": "reach_k1"})
        )
        reach_at_kmax = (
            goal_reach.filter(pl.col("k") == mode_k_values[-1])
            .select("goal", "reach_rate")
            .rename({"reach_rate": "reach_kmax"})
        )
        goal_clusters = reach_at_k1.join(reach_at_kmax, on="goal").with_columns(
            pl.when(pl.col("reach_k1") > 0.7)
            .then(pl.lit("easy"))
            .when(pl.col("reach_kmax") < 0.3)
            .then(pl.lit("hard"))
            .otherwise(pl.lit("scalable"))
            .alias("cluster")
        )

        cluster_linestyles = {"easy": "-", "scalable": "--", "hard": ":"}
        for cluster, ls in cluster_linestyles.items():
            goals_in = goal_clusters.filter(pl.col("cluster") == cluster)["goal"].to_list()
            suffix = "" if is_single else f", {n_goals}×{n_trials}"
            for gi, g in enumerate(goals_in):
                row = goal_reach.filter(pl.col("goal") == g).sort("k")
                ax.plot(
                    row["k"],
                    row["reach_rate"] * 100,
                    color=style["color"],
                    linestyle=ls,
                    alpha=0.5,
                    linewidth=1,
                    label=f"{mode_name} ({cluster}{suffix})" if gi == 0 else None,
                )

    ax.set_xlabel("k (number of guides)")
    ax.set_ylabel("Reachability (%)")
    ax.set_title(
        f"Per-goal reachability vs k [{ds_name}]\n(solid=easy, dashed=scalable, dotted=hard)"
    )
    configure_k_axis(ax, k_values)
    ax.set_ylim(-5, 105)
    ax.legend(fontsize=7)

    finish_fig(fig, "Goal Reachability", ds_name, n_goals=n_goals, n_trials=n_trials)

## Reachability tradeoff

Node count is `max(trial_nodes, guide_egraph_nodes)` and time includes `guide_eqsat_time`, so both reflect the true cost of the guided run.

In [ ]:
for ds_name in ds_names:
    mode_dfs = dataset_frames[ds_name]
    n_goals, n_trials = dataset_n[ds_name]
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for i, (mode_name, _run_dir, is_single) in enumerate(SAMPLING_MODES):
        style = mode_style(i, is_single)
        df = mode_dfs[mode_name]

        tradeoff = (
            df.group_by("k")
            .agg(
                pl.col("reached").mean().alias("reachability"),
                pl.col("nodes").mean().alias("avg_nodes"),
                pl.col("total_time").mean().alias("avg_time"),
            )
            .sort("k")
            .drop_nulls(["avg_nodes", "avg_time"])
        )
        ks = tradeoff["k"].to_list()
        reach = tradeoff["reachability"].to_list()

        axes[0].plot(
            tradeoff["avg_nodes"], tradeoff["reachability"] * 100, label=mode_name, **style
        )
        if not is_single:
            annotate_k_points(axes[0], tradeoff["avg_nodes"].to_list(), reach, ks, scale=100)

        axes[1].plot(
            tradeoff["avg_time"] * 1000, tradeoff["reachability"] * 100, label=mode_name, **style
        )
        if not is_single:
            annotate_k_points(
                axes[1], [t * 1000 for t in tradeoff["avg_time"].to_list()], reach, ks, scale=100
            )

    axes[0].set_xlabel("Avg nodes, incl. guide egraph (all trials)")
    axes[0].set_ylabel("Reachability (%)")
    axes[0].set_title("Reachability vs node cost (incl. guide egraph)")
    axes[0].legend()

    axes[1].set_xlabel("Avg time, incl. guide eqsat (ms)")
    axes[1].set_ylabel("Reachability (%)")
    axes[1].set_title("Reachability vs time cost (incl. guide eqsat)")
    axes[1].legend()

    finish_fig(fig, "Reachability Tradeoff", ds_name, n_goals=n_goals, n_trials=n_trials)

## Wins Above Replacement (WAR)

WAR measures how much better each strategy is compared to the full eqsat baseline (no guides).

- **nodes WAR**: baseline nodes − mean guided nodes (positive = guided uses fewer nodes; guided nodes = `max(trial_nodes, guide_egraph_nodes)`)
- **time WAR**: baseline time − mean guided time (positive = guided is faster; guided time includes `guide_eqsat_time`)

Both are computed per strategy × k. The baseline comes from each run's seed-terms
`goal_egraph` (see the baseline cell).

In [ ]:
for ds_name in ds_names:
    mode_dfs = dataset_frames[ds_name]
    mode_dfs = {k: v.filter(pl.col("reached")) for k, v in mode_dfs.items()}
    n_goals, n_trials = dataset_n[ds_name]
    war_rows = []

    for mode_name, _run_dir, _is_single in SAMPLING_MODES:
        df = mode_dfs[mode_name]
        for row in compute_war_agg(df, baselines[ds_name], ["nodes", "total_time"]):
            war_rows.append({"mode": mode_name, **row})

    war_df = pl.DataFrame(war_rows).sort("mode", "k")

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for i, (mode_name, _run_dir, is_single) in enumerate(SAMPLING_MODES):
        style = mode_style(i, is_single)
        sub = war_df.filter(pl.col("mode") == mode_name).sort("k")
        if sub.is_empty():
            continue
        ks = sub["k"]

        if is_single:
            axes[0].plot(ks, sub["nodes_war"], label=mode_name, **style)
            axes[1].plot(ks, sub["total_time_war"] * 1000, label=mode_name, **style)
        else:
            plot_mean_with_best_dotted(
                axes[0],
                ks,
                sub["nodes_war"],
                sub["best_nodes_war"],
                style,
                label=mode_name,
                best_label=f"{mode_name} best (min)",
            )
            plot_mean_with_best_dotted(
                axes[1],
                ks,
                sub["total_time_war"] * 1000,
                sub["best_total_time_war"] * 1000,
                style,
                label=mode_name,
                best_label=f"{mode_name} best (min)",
            )

    for ax in axes:
        ax.axhline(0, color="black", linewidth=0.8, linestyle="-", alpha=0.5)
        configure_k_axis(ax, k_values)
        ax.set_xlabel("k (number of guides)")
        ax.legend(fontsize=7)

    axes[0].set_ylabel("nodes WAR  (baseline - guided, higher is better)")
    axes[0].set_title("Nodes WAR vs k")
    axes[1].set_ylabel("time WAR  (ms, baseline - guided, higher is better)")
    axes[1].set_title("Time WAR vs k")

    finish_fig(fig, "Wins Above Replacement (WAR)", ds_name, n_goals=n_goals, n_trials=n_trials)
    print(war_df)